In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from pathlib import Path
from sklearn.model_selection import train_test_split

# Корневая папка проекта
BASE_DIR = Path.cwd()

if not (BASE_DIR / "data" / "processed" / "sms_clean.csv").exists():
    BASE_DIR = BASE_DIR.parent

# Загружаем очищенный датасет
DATA_PATH = BASE_DIR / "data" / "processed" / "sms_clean.csv"

df = pd.read_csv(DATA_PATH)

# Сбрасываем индекс
df = df.reset_index(drop=True)

print("=== SMS GUARD — ЗАДАНИЕ №6 ===")

print("\nКоличество сообщений:", len(df))

print("\nРаспределение классов:")
print(df["label"].value_counts())

print("\nПервые 5 записей:")
display(df.head())

=== SMS GUARD — ЗАДАНИЕ №6 ===

Количество сообщений: 5103

Распределение классов:
label
0    4501
1     602
Name: count, dtype: int64

Первые 5 записей:


,id,text_raw,text_clean,label
0,1,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,0
1,2,Ok lar... Joking wif u oni...,ok lar joking wif u oni,0
2,3,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in <num> a wkly comp to win fa cup ...,1
3,4,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,0
4,5,"Nah I don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives arou...,0


In [3]:
# ==============================================
# РАЗДЕЛЕНИЕ ДАННЫХ TRAIN / VALIDATION / TEST
# ==============================================

RANDOM_STATE = 42

# Индексы всех сообщений
indices = df.index.to_numpy()

# Первое разделение:
# 70% обучение, 30% временная выборка

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

# Второе разделение:
# 15% валидация, 15% тест

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=df.loc[temp_idx, "label"]
)

# Создаём три выборки
train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()
test_df = df.iloc[test_idx].copy()

print("=== РЕЗУЛЬТАТ РАЗДЕЛЕНИЯ ===")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nОбщее количество:")
print(len(train_df) + len(val_df) + len(test_df))

=== РЕЗУЛЬТАТ РАЗДЕЛЕНИЯ ===
Train: 3572
Validation: 765
Test: 766

Общее количество:
5103


In [4]:
# ==============================================
# ПРОВЕРКА РАСПРЕДЕЛЕНИЯ КЛАССОВ
# ==============================================

datasets = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}

for name, dataset in datasets.items():

    print(f"\n=== {name} ===")

    print("Количество SMS:", len(dataset))

    print("\nКоличество по классам:")
    print(dataset["label"].value_counts())

    print("\nПроцентное распределение:")
    print(
        (
            dataset["label"]
            .value_counts(normalize=True)
            * 100
        ).round(2)
    )


=== Train ===
Количество SMS: 3572

Количество по классам:
label
0    3151
1     421
Name: count, dtype: int64

Процентное распределение:
label
0    88.21
1    11.79
Name: proportion, dtype: float64

=== Validation ===
Количество SMS: 765

Количество по классам:
label
0    675
1     90
Name: count, dtype: int64

Процентное распределение:
label
0    88.24
1    11.76
Name: proportion, dtype: float64

=== Test ===
Количество SMS: 766

Количество по классам:
label
0    675
1     91
Name: count, dtype: int64

Процентное распределение:
label
0    88.12
1    11.88
Name: proportion, dtype: float64


In [5]:
# ==============================================
# СОХРАНЕНИЕ ИНДЕКСОВ РАЗДЕЛЕНИЯ
# ==============================================

SPLIT_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "sms_split.json"
)

# Проверяем отсутствие пересечений
train_set = set(train_idx)
val_set = set(val_idx)
test_set = set(test_idx)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

# Проверяем, что все записи распределены
assert len(train_set | val_set | test_set) == len(df)

# Подготавливаем данные
split_data = {
    "random_state": RANDOM_STATE,
    "train": train_idx.tolist(),
    "validation": val_idx.tolist(),
    "test": test_idx.tolist()
}

# Создаём папку при необходимости
SPLIT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Сохраняем JSON
with open(SPLIT_PATH, "w", encoding="utf-8") as file:

    json.dump(
        split_data,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Разделение успешно сохранено!")
print("Путь:", SPLIT_PATH)

print("\nПроверка:")
print("Пересечений между выборками нет.")
print("Всего распределено SMS:", len(df))

Разделение успешно сохранено!
Путь: D:\SMS_Guard_Task_02_Starter\sms_guard\data\processed\sms_split.json

Проверка:
Пересечений между выборками нет.
Всего распределено SMS: 5103


## Шаг 5. TF-IDF и ручные признаки

Пайплайн из `ml/features.py` принимает исходные тексты `text_raw`, очищает их **той же** функцией `normalize_sms`, затем формирует три ветви признаков:

1. Слова — TF-IDF для униграмм и биграмм, `ngram_range=(1,2)`, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`.
2. Символьные N-граммы — TF-IDF, `analyzer='char_wb'`, `ngram_range=(3,5)`.
3. Ручные признаки — длина, доля заглавных букв, число `!`, наличие URL, денег, телефона, число сработавших групп социальной инженерии. Числовые признаки масштабируются.

**Важно:** `fit_transform` вызывается только на обучающей выборке. Валидация и тест используют только `transform`. Тестовую выборку не используем для выбора признаков, параметров или порога.

In [6]:
import sys
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from ml.features import make_sms_features_pipeline

sms_features_pipeline = make_sms_features_pipeline()

# Берём сырые тексты: нормализация уже включена внутрь Pipeline.
train_texts = train_df["text_raw"].fillna("")
val_texts = val_df["text_raw"].fillna("")
test_texts = test_df["text_raw"].fillna("")

X_train = sms_features_pipeline.fit_transform(train_texts)  # обучение только train
X_val = sms_features_pipeline.transform(val_texts)           # без fit
X_test = sms_features_pipeline.transform(test_texts)        # без fit

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

assert X_train.shape[0] == len(train_df)
assert X_val.shape[0] == len(val_df)
assert X_test.shape[0] == len(test_df)
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
print("✅ Размерности согласованы, fit на test/val не выполнялся")

Train: (3572, 29651)
Validation: (765, 29651)
Test: (766, 29651)
✅ Размерности согласованы, fit на test/val не выполнялся


## Шаг 6. Топ-15 словесных TF-IDF признаков по классам (только train)

Для анализа вычисляем средний TF-IDF по обучающим SMS в каждом классе. Частота и средний TF-IDF — разные показатели: TF-IDF дополнительно снижает вес часто встречающихся слов по всему корпусу.

In [7]:
word_pipeline = sms_features_pipeline.named_steps["features"].named_transformers["words"]
word_vectorizer = word_pipeline.named_steps["tfidf"]
word_X_train = word_pipeline.transform(train_texts)  # без fit
word_names = word_vectorizer.get_feature_names_out()

for klass, title in [(0, "Обычные SMS"), (1, "Спам-SMS")]:
    avg_tfidf = np.asarray(word_X_train[y_train == klass].mean(axis=0)).ravel()
    top_idx = np.argsort(avg_tfidf)[-15:][::-1]
    table = pd.DataFrame({
        "Признак": word_names[top_idx],
        "Средний TF-IDF": np.round(avg_tfidf[top_idx], 5),
    })
    print(f"Топ-15: {title}")
    display(table)

Топ-15: Обычные SMS


,Признак,Средний TF-IDF
0,you,0.03233
1,to,0.02471
2,the,0.02065
3,in,0.01872
4,me,0.01758
5,and,0.01703
6,it,0.01618
7,my,0.01611
8,is,0.01596
9,num,0.01595


Топ-15: Спам-SMS


,Признак,Средний TF-IDF
0,num,0.08713
1,phone,0.05567
2,to,0.04860
3,call,0.04136
4,money,0.04110
5,call phone,0.03293
6,your,0.03209
7,to num,0.03183
8,free,0.03108
9,or,0.02487


## Объяснение TF-IDF и N-грамм

**TF-IDF (Term Frequency — Inverse Document Frequency)** представляет текст как вектор числовых весов. Частота слова в одном документе (TF) учитывает, насколько заметно оно внутри этого SMS, а обратная документная частота (IDF) уменьшает значимость слов, часто встречающихся в разных сообщениях. Благодаря этому признаки вроде `claim` или `prize` могут получать полезный вес, если они характерны для некоторой части сообщений, а обычные общеупотребительные слова влияют меньше. В нашей реализации параметр `sublinear_tf=True` заменяет обычную частоту логарифмически сглаженной, чтобы повторение одного слова много раз не давало ему чрезмерного веса.

**N-граммы слов** — это цепочки из N слов. Униграмма соответствует одному слову (`prize`), а биграмма — паре (`claim prize`). Пары слов позволяют учитывать небольшой контекст, что особенно важно для SMS: отдельное слово «код» не обязательно опасно, но сочетание «сообщите код» может служить сигналом социальной инженерии. В задаче используется `ngram_range=(1,2)`.

**Символьные N-граммы** строятся из последовательностей символов. Диапазон от трёх до пяти символов помогает находить части слов даже при опечатках, сокращениях и намеренном искажении написания. Вариант `char_wb` учитывает границы слов, дополняя их пробелами. Поэтому признаки менее зависимы от точного написания слова.

Текстовые признаки объединяются с ручными характеристиками (длина, регистр, восклицательные знаки, наличие URL и денежных сумм, число групп социальной инженерии). Эти признаки могут дополнять TF-IDF, но их пользу нужно проверять на валидации. Признаки образуют разреженную матрицу, а подготовка включена в единый `Pipeline`, который можно сохранить вместе с моделью.

**Защита от утечки данных:** модель словаря и IDF обучается только на `train`. Для `validation` и `test` выполняется `transform`. Значения неизвестных в обучении слов не дополняют словарь при проверке. Тестовая выборка остаётся нетронутой до финальной оценки в задании №8.

**Контроль перед заданием №7:** сохраните ноутбук и `sms_split.json`; затем запустите `python -m pytest -q`. Для обучения первой модели в задании №7 используйте `make_sms_features_pipeline()` внутри классификационного Pipeline — так нормализация и словарь сохранятся вместе с моделью.